# Mesh Simplification

In this notebook, I wish to explore the basics of Mesh Simplification. The primary algorithms available to us are the Edge Collapse algorithm (used in modifiers like Decimate in Blender), and certain Machine Learning techniques. Let's explore

In [72]:
import pandas as pd
import numpy as np
import pymeshlab 

We import a mesh using the `MeshSet()` class in PyMeshLab.

In [73]:
ms = pymeshlab.MeshSet()
ms.load_new_mesh('../../../models/foot/human-foot.obj')

Now that we have the mesh in memory, we should be able to view basic attributes relating to it. Here, we see the list of vertices we have available

In [74]:
# Need to create a reference to the current mesh
m = ms.current_mesh()
v_matrix = m.vertex_matrix()

v_matrix

array([[ 0.059608,  0.383419, -0.047925],
       [-0.030599, -0.01601 , -0.020009],
       [ 0.052395,  0.402345,  0.102574],
       ...,
       [ 0.174218,  0.030867,  0.461069],
       [ 0.163158,  0.035477,  0.397872],
       [-0.056806,  0.012867,  0.570864]], shape=(800, 3))

We see the familiar (800,3) shaped array implying 800 vertices each with x,y,z coordinates.

We can even convert to dataframe.

In [75]:
vertex_df = pd.DataFrame(v_matrix, columns=["X", "Y", "Z"])
vertex_df.head()

,X,Y,Z
0,0.059608,0.383419,-0.047925
1,-0.030599,-0.016010,-0.020009
2,0.052395,0.402345,0.102574
3,0.049393,-0.015811,-0.022423
4,-0.061567,0.421855,-0.066727


Let's apply a filter. In [mesh-decimation.md markdown file](../mesh-simplification.md) we discussed working with MeshLab through the GUI. In that document, we applied a Decimation with Quadric error, while maintaining the original mesh vertices. Let's do the same through the Python API.

In the [API](https://pymeshlab.readthedocs.io/en/latest/filter_list.html#meshing_decimation_quadric_edge_collapse) we see that this function is called as a method on our MeshSet(). Let's do this, using the default arguments, except for `optimalplacement` which we shall set to `False`. This argument controls the placement of our decimated vertices. In this case, we want our decimated vertices to maintain their original location coordinates.

In [76]:
ms.meshing_decimation_quadric_edge_collapse(optimalplacement=False)

Let's see what the result looks like. We're expecting a reduction in the number of vertices.

In [77]:
m = ms.current_mesh()
v_matrix_decimate = m.vertex_matrix()

v_matrix_decimate

array([[ 0.059608,  0.383419, -0.047925],
       [-0.030599, -0.01601 , -0.020009],
       [ 0.052395,  0.402345,  0.102574],
       ...,
       [ 0.100379, -0.002407,  0.430997],
       [ 0.130661,  0.005967,  0.426889],
       [ 0.174218,  0.030867,  0.461069]], shape=(403, 3))

Indeed we observe that our mesh has reduced vertices. Let's convert to dataframe and see what these vertices look like.

In [78]:
vertex_df_decimate = pd.DataFrame(v_matrix_decimate, columns = ["X", "Y", "Z"])
vertex_df_decimate.head()

,X,Y,Z
0,0.059608,0.383419,-0.047925
1,-0.030599,-0.016010,-0.020009
2,0.052395,0.402345,0.102574
3,0.049393,-0.015811,-0.022423
4,-0.061567,0.421855,-0.066727


In order to see whether our vertices in `vertex_df_decimate` match with the original, I will apply the following transformations.

1) Create a new column containing the tuple of (x,y,z) coordinates for each dataframe.
2) Merge dataframes based on this column (innerjoin).
3) Verfiy that the number of rows in each case is the same (i.e. no unmatched rows)

In [79]:
vertex_df['coords'] = vertex_df[["X", "Y", "Z"]].apply(tuple, axis=1)
vertex_df.head()

,X,Y,Z,coords
0,0.059608,0.383419,-0.047925,"(0.059608, 0.383419, -0.047925)"
1,-0.030599,-0.016010,-0.020009,"(-0.030599, -0.01601, -0.020009)"
2,0.052395,0.402345,0.102574,"(0.052395, 0.402345, 0.102574)"
3,0.049393,-0.015811,-0.022423,"(0.049393, -0.015811, -0.022423)"
4,-0.061567,0.421855,-0.066727,"(-0.061567, 0.421855, -0.066727)"


In [80]:
vertex_df_decimate['coords'] = vertex_df_decimate[["X", "Y", "Z"]].apply(tuple, axis=1)
vertex_df_decimate.head()

,X,Y,Z,coords
0,0.059608,0.383419,-0.047925,"(0.059608, 0.383419, -0.047925)"
1,-0.030599,-0.016010,-0.020009,"(-0.030599, -0.01601, -0.020009)"
2,0.052395,0.402345,0.102574,"(0.052395, 0.402345, 0.102574)"
3,0.049393,-0.015811,-0.022423,"(0.049393, -0.015811, -0.022423)"
4,-0.061567,0.421855,-0.066727,"(-0.061567, 0.421855, -0.066727)"


In [81]:
merged_df_vertex = pd.merge(vertex_df, vertex_df_decimate, how="outer", on="coords")
len(merged_df_vertex)

800

Perfect. We observe that each vertex in our 2 meshes are exactly aligned. To drive this point home, I shall repeat the above experiment with the same filter except this time I wont set the `optimalplacement` parameter to False. In this case, I expect that the model will create new vertices in our decimated mesh to maintain the overall shape. While this new mesh may look closer to the old one, it will likely create new vertices, implying that our total number of rows in the 

In [82]:
ms_2 = pymeshlab.MeshSet()
ms_2.load_new_mesh('../../../models/foot/human-foot.obj')

ms_2.meshing_decimation_quadric_edge_collapse(optimalplacement=True) #reverting this to True

m_test = ms_2.current_mesh()
v_matrix_test = m_test.vertex_matrix()

vertex_df_test = pd.DataFrame(v_matrix_test, columns = ["X", "Y", "Z"])

vertex_df_test['coords'] = vertex_df_test[["X", "Y", "Z"]].apply(tuple, axis=1)

test_df_vertex = pd.merge(vertex_df, vertex_df_test, how="outer", on="coords")
len(test_df_vertex)

975

As expected, we see an increase in the number of vertices, implying that we are creating new ones.

Let's work with the faces now and see whats going on. We access the faces in the mesh class by using the following code.

In [ ]:
m = ms.current_mesh()
v_matrix = m.vertex_matrix()

v_matrix